# Rationality TTS — Colab workspace (Drama 3)

This notebook runs the **SvelteKit app + Fish Audio Drama 3** on a Colab GPU and
exposes it through a public URL. The app's state is stored in the data repo
(`denlukia/rationality-tts-data`), so you can commit your work before the
runtime expires and resume later.

**Read before running:**
- Colab's FAQ forbids "web service offerings not related to interactive compute"
  and "connecting to remote proxies" on managed runtimes. Running this notebook
  is a policy violation; the runtime can be terminated without warning. The app
  auto-commits every 5 minutes so a kill costs at most a few minutes of work.
- Drama 3 needs an **A100 with 80 GB** (the int4 checkpoint peaks at ~57 GB of
  VRAM). Free T4 / L4 / A100-40 will not work. Open *Runtime → Change runtime
  type → A100* (high RAM) before running.
- The first cold start takes ~10–20 minutes (model download + SGLang install).
  With Pro+ continuous execution a runtime lives up to 24 h, so start it once
  per working day.

In [ ]:
!nvidia-smi | head -20
import subprocess, sys
out = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], text=True)
print(out)
name, mem = out.strip().split(',')
mem_gb = float(mem.strip().split()[0]) / 1024
if mem_gb < 70:
    raise SystemExit(f'Need an A100 with ~80 GB VRAM, got {mem.strip()}. Change the runtime type.')
print('GPU OK:', name.strip(), mem.strip())

In [ ]:
REPO = 'https://github.com/denlukia/rationality-tts'
!rm -rf /content/rationality-tts
!git clone --depth 1 --recurse-submodules "$REPO" /content/rationality-tts
%cd /content/rationality-tts
!ls

In [ ]:
# Node + bun
!curl -fsSL https://bun.sh/install | bash
import os
os.environ['PATH'] = '/root/.bun/bin:' + os.environ['PATH']
!bun --version
!bun install
!bun run build
print('app built')

In [ ]:
# Python deps (no Holos — Drama only)
!pip install -q fastapi uvicorn numpy soundfile python-multipart huggingface_hub
!pip install -q "git+https://github.com/patriotyk/ukrainian-word-stress.git"
!pip install -q "git+https://github.com/patriotyk/ipa-uk.git"
print('python deps ready')

In [ ]:
# SGLang-Omni runtime (heavy, ~5-10 min)
!rm -rf /content/sglang-omni
!git clone --depth 1 https://github.com/sgl-project/sglang-omni.git /content/sglang-omni
%cd /content/sglang-omni
!uv pip install --prerelease=allow -q -e '.[fishaudio]'
%cd /content/rationality-tts
print('sglang-omni installed')

In [ ]:
import os, time, subprocess, urllib.request
# Start SGLang-Omni with the int4 checkpoint on port 8000.
log = open('/tmp/sgl.log', 'w')
proc = subprocess.Popen(
    ['sgl-omni', 'serve', '--model-path', 'fishaudio/drama-3-int4', '--port', '8000'],
    stdout=log, stderr=subprocess.STDOUT,
    cwd='/content/sglang-omni',
)
# Wait for readiness.
for _ in range(180):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=3)
        print('sglang-omni ready')
        break
    except Exception:
        time.sleep(2)
else:
    print(open('/tmp/sgl.log').read()[-2000:])
    raise SystemExit('sglang-omni did not start')

In [ ]:
import os, subprocess, time
os.environ['TTS_URL'] = 'http://127.0.0.1:8767'
os.environ['TTS_ENGINE'] = 'drama'
os.environ['DRAMA_BACKEND'] = 'remote'
os.environ['DRAMA_REMOTE_URL'] = 'http://127.0.0.1:8000'
os.environ['SESSION_REQUIRE_AUTH'] = '1'
os.environ['SESSION_BUDGET_SEC'] = '82800'   # 23 h
os.environ['GITHUB_REPO'] = 'denlukia/rationality-tts-data'
os.environ['DATA_DIR'] = '/content/rationality-tts/data'
os.environ['CONTENT_DIR'] = '/content/rationality-tts/content/books'

# Sidecar (no Holos deps needed)
sidecar = subprocess.Popen(
    ['python', 'tts/server.py'],
    cwd='/content/rationality-tts', stdout=open('/tmp/sidecar.log','w'), stderr=subprocess.STDOUT,
)
# App
app = subprocess.Popen(
    ['node', 'build'],
    cwd='/content/rationality-tts', stdout=open('/tmp/app.log','w'), stderr=subprocess.STDOUT,
    env={**os.environ, 'PORT':'3000', 'HOST':'0.0.0.0'},
)
time.sleep(6)
print('sidecar:', open('/tmp/sidecar.log').read()[-500:])
print('app:', open('/tmp/app.log').read()[-500:])

In [ ]:
# cloudflared tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess, re, time
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:3000'],
    stdout=open('/tmp/cf.log','w'), stderr=subprocess.STDOUT,
)
for _ in range(60):
    time.sleep(1)
    txt = open('/tmp/cf.log').read()
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', txt)
    if m:
        url = m.group(0)
        print('\n===== OPEN THIS URL =====')
        print(url)
        print('=========================')
        print('Passcode: paste your GitHub fine-grained token on the login screen.')
        break
else:
    print(open('/tmp/cf.log').read()[-1000:])